# Multimodal Content Moderation — Text-Image Consistency Detection

**Task:** classify product listings as `compliant` (genuine text-image pair) or `non-compliant` (image swapped across an unrelated category — simulating a mislabeled/recycled-image listing).

**Why multimodal:** neither text alone nor image alone can detect a mismatch — each looks fine in isolation. Only a model reasoning over *both together* can.

**Design principles for this rebuild:**
- All three models (text-only, image-only, fusion) train/evaluate on the **exact same subsample and split** — a fair ablation requires this.
- Every section **saves its output to Drive immediately** (labels, split, model weights, metrics) — a disconnect only costs you the section in progress, nothing earlier.
- Metrics accumulate in `metrics.json` on Drive — the final summary table is generated from real saved numbers, not manually typed in.

**Run order:** top to bottom, once. If you get disconnected, just re-run from wherever it stopped — earlier sections' outputs are already saved and will be reloaded, not recomputed.


## 0. Setup & GPU check

**Before running anything: Runtime → Change runtime type → T4 GPU.** The cell below will warn you loudly if you're on CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS if your folder is named differently
DATA_DIR = '/content/drive/MyDrive/moderation-project/data'

import torch
if not torch.cuda.is_available():
    print("\n" + "="*60)
    print("WARNING: No GPU detected. Go to Runtime > Change runtime")
    print("type > T4 GPU, then Runtime > Restart session, then re-run")
    print("from this cell. Training on CPU will be very slow.")
    print("="*60 + "\n")
else:
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")


In [ ]:
!pip install -q torch torchvision scikit-learn pandas pillow transformers


In [ ]:
import os, json

# Config -- fixed across the whole notebook so every model sees the same data
SEED = 42
N_SAMPLES = 20000       # fixed subsample size used by ALL THREE models for a fair comparison
SWAP_FRAC = 0.25        # fraction of listings given a mismatched image
TEXT_EPOCHS = None      # n/a, LogReg trains in one shot
IMAGE_EPOCHS = 4
FUSION_EPOCHS = 4
IMAGES_DIR = "/content/images_train_raw/images_train"

METRICS_PATH = f"{DATA_DIR}/metrics.json"

def load_metrics():
    if os.path.exists(METRICS_PATH):
        with open(METRICS_PATH) as f:
            return json.load(f)
    return {}

def save_metric(name, value):
    m = load_metrics()
    m[name] = value
    with open(METRICS_PATH, "w") as f:
        json.dump(m, f, indent=2)
    print(f"Saved metric '{name}' = {value:.4f} -> {METRICS_PATH}")

print("Config set. Current saved metrics (if any):", load_metrics())


## 1. Unzip images (idempotent — skips if already done this session)

In [ ]:
if not os.path.exists(IMAGES_DIR):
    if not os.path.exists('/content/images_train_raw'):
        !unzip -q "{DATA_DIR}/images.zip" -d /content/images_train_raw
    print("Contents after unzip:")
    !find /content/images_train_raw -maxdepth 2 -type d
    # If IMAGES_DIR above doesn't match the real path printed here, edit the
    # IMAGES_DIR variable in the config cell above and re-run from there.
else:
    print(f"Images already available at {IMAGES_DIR}, skipping unzip.")


## 2. Build mismatch labels + fixed subsample/split

This is the ONE split every model below uses. Saved to Drive so re-running the notebook (or resuming after a disconnect) reloads the same split instead of regenerating a different random one.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SPLIT_PATH = f"{DATA_DIR}/fixed_split.csv"

CATEGORY_GROUPS = {
    "books_media": [10, 2280, 2403, 2705, 2522],
    "toys_games_figures": [40, 50, 60, 1140, 1160, 1180, 1280, 1281, 1300,
                            1301, 1302, 1320, 2462, 2905],
    "home_furniture_garden": [1560, 1920, 1940, 2060, 2220, 2582, 2583, 2585],
}
CODE_TO_GROUP = {code: g for g, codes in CATEGORY_GROUPS.items() for code in codes}

def build_labels(input_dir, swap_frac, seed):
    rng = np.random.default_rng(seed)
    X = pd.read_csv(f"{input_dir}/X_train_update.csv", index_col=0)
    Y = pd.read_csv(f"{input_dir}/Y_train_CVw08PX.csv", index_col=0)
    df = X.join(Y).reset_index(drop=True)
    df["group"] = df["prdtypecode"].map(CODE_TO_GROUP)
    assert df["group"].isna().sum() == 0, "Unmapped prdtypecode found."

    n = len(df)
    n_swap = int(n * swap_frac)
    swap_idx = rng.choice(n, size=n_swap, replace=False)

    df["label"] = 0
    df["orig_imageid"] = df["imageid"]
    df["orig_productid"] = df["productid"]
    df["productid_for_image"] = df["orig_productid"]

    for idx in swap_idx:
        own_group = df.at[idx, "group"]
        donor_pool = df.index[df["group"] != own_group]
        donor_idx = rng.choice(donor_pool)
        df.at[idx, "imageid"] = df.at[donor_idx, "orig_imageid"]
        df.at[idx, "productid_for_image"] = df.at[donor_idx, "orig_productid"]
        df.at[idx, "label"] = 1

    df["productid_for_image"] = df["productid_for_image"].astype(df["orig_productid"].dtype)
    df["description"] = df["description"].fillna("")
    df["text"] = (df["designation"] + " " + df["description"]).str.strip()
    df["fname"] = "image_" + df["imageid"].astype(str) + "_product_" + df["productid_for_image"].astype(str) + ".jpg"
    return df

if os.path.exists(SPLIT_PATH):
    print("Fixed split already exists, loading from Drive (not regenerating)...")
    full_df = pd.read_csv(SPLIT_PATH)
else:
    print("Building fresh labels + subsample + split...")
    df = build_labels(DATA_DIR, SWAP_FRAC, SEED)

    # Only keep rows whose image file actually exists locally
    available = set(os.listdir(IMAGES_DIR))
    df = df[df["fname"].isin(available)].reset_index(drop=True)
    print(f"Rows with available images: {len(df)}")

    # Fixed stratified subsample -- same size/seed used by every model below
    df_sample = df.sample(n=min(N_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)

    train_df, val_df = train_test_split(
        df_sample, test_size=0.2, stratify=df_sample["label"], random_state=SEED
    )
    train_df["split"] = "train"
    val_df["split"] = "val"
    full_df = pd.concat([train_df, val_df], ignore_index=True)
    full_df.to_csv(SPLIT_PATH, index=False)
    print(f"Saved fixed split -> {SPLIT_PATH}")

train_df = full_df[full_df["split"] == "train"].reset_index(drop=True)
val_df = full_df[full_df["split"] == "val"].reset_index(drop=True)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)}")
print(f"Train label balance:\n{train_df[\'label\'].value_counts(normalize=True)}")

n_pos = (train_df["label"] == 1).sum()
n_neg = (train_df["label"] == 0).sum()
class_weight = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float32)


## 3. Text-only baseline (expected: near-chance — text is untouched by the image swap)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
X_train_vec = vectorizer.fit_transform(train_df["text"])
X_val_vec = vectorizer.transform(val_df["text"])

clf_text = LogisticRegression(max_iter=1000, class_weight="balanced")
clf_text.fit(X_train_vec, train_df["label"])
preds = clf_text.predict(X_val_vec)

print(classification_report(val_df["label"], preds, target_names=["compliant", "non-compliant"]))
f1_text = f1_score(val_df["label"], preds, pos_label=1)
print(f"F1 (non-compliant): {f1_text:.4f}")
save_metric("text_only_f1", f1_text)


## 4. Image-only baseline (pretrained ResNet18, fine-tuned — expected: near-chance)

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weight = class_weight.to(device)

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class ImageDataset(Dataset):
    def __init__(self, df, images_dir, transform):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.images_dir, row["fname"])).convert("RGB")
        return self.transform(img), int(row["label"])

train_loader = DataLoader(ImageDataset(train_df, IMAGES_DIR, tf), batch_size=32, shuffle=True)
val_loader = DataLoader(ImageDataset(val_df, IMAGES_DIR, tf), batch_size=32, shuffle=False)

img_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
img_model.fc = nn.Linear(img_model.fc.in_features, 2)
img_model = img_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weight)
optimizer = torch.optim.Adam(img_model.parameters(), lr=1e-4)

for epoch in range(IMAGE_EPOCHS):
    img_model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(img_model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    print(f"Epoch {epoch+1}/{IMAGE_EPOCHS} - loss: {total_loss/len(train_df):.4f}")
    torch.save(img_model.state_dict(), f"{DATA_DIR}/image_baseline_epoch{epoch+1}.pt")

img_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        preds = img_model(imgs.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds); all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=["compliant", "non-compliant"]))
f1_image = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)
print(f"F1 (non-compliant): {f1_image:.4f}")
save_metric("image_only_f1", f1_image)

torch.save(img_model.state_dict(), f"{DATA_DIR}/image_baseline_final.pt")


## 5. Fusion model (text + image jointly)

**Architecture note:** match/mismatch detection is a *comparison* task, not a "predict from a pile of features" task. Naively concatenating two embeddings gives a classifier no explicit signal about how the two relate. Here, both embeddings are projected to a shared dimension, and the classifier receives `[text, image, |text-image|, text*image]` — the difference/product terms give it direct access to how aligned or misaligned the pair is (the standard trick from NLI/sentence-matching models).

**Speed note:** DistilBERT is frozen, so its embeddings are precomputed once up front rather than recomputed every batch/epoch — this was the main hidden bottleneck in earlier runs.

In [ ]:
from transformers import AutoTokenizer, AutoModel

TEXT_MODEL_NAME = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(device).eval()
for p in text_encoder.parameters():
    p.requires_grad = False

def embed_texts(df_subset, batch_size=64, max_len=64):
    embeddings = []
    texts = df_subset["text"].tolist()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, truncation=True, padding="max_length",
                             max_length=max_len, return_tensors="pt").to(device)
            out = text_encoder(**enc)
            embeddings.append(out.last_hidden_state[:, 0, :].cpu())
    return torch.cat(embeddings, dim=0)

TRAIN_EMB_PATH = f"{DATA_DIR}/train_text_emb.pt"
VAL_EMB_PATH = f"{DATA_DIR}/val_text_emb.pt"

if os.path.exists(TRAIN_EMB_PATH) and os.path.exists(VAL_EMB_PATH):
    print("Loading cached text embeddings from Drive...")
    train_text_emb = torch.load(TRAIN_EMB_PATH)
    val_text_emb = torch.load(VAL_EMB_PATH)
else:
    print("Precomputing text embeddings (one-time cost)...")
    train_text_emb = embed_texts(train_df)
    val_text_emb = embed_texts(val_df)
    torch.save(train_text_emb, TRAIN_EMB_PATH)
    torch.save(val_text_emb, VAL_EMB_PATH)
print("Done.")


In [ ]:
class FusionDataset(Dataset):
    def __init__(self, df, text_embs, images_dir, transform):
        self.df = df.reset_index(drop=True)
        self.text_embs = text_embs
        self.images_dir = images_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.images_dir, row["fname"])).convert("RGB")
        img = self.transform(img)
        return img, self.text_embs[idx], int(row["label"])

class FusionModel(nn.Module):
    def __init__(self, img_backbone, text_dim=768, img_dim=512, shared_dim=256, hidden=256):
        super().__init__()
        self.img_backbone = img_backbone
        self.img_backbone.fc = nn.Identity()
        self.text_proj = nn.Linear(text_dim, shared_dim)
        self.img_proj = nn.Linear(img_dim, shared_dim)
        self.classifier = nn.Sequential(
            nn.Linear(shared_dim * 4, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, 2)
        )
    def forward(self, imgs, text_emb):
        img_feat = self.img_backbone(imgs)
        t = self.text_proj(text_emb)
        v = self.img_proj(img_feat)
        interaction = torch.cat([t, v, torch.abs(t - v), t * v], dim=1)
        return self.classifier(interaction)

train_fds = FusionDataset(train_df, train_text_emb, IMAGES_DIR, tf)
val_fds = FusionDataset(val_df, val_text_emb, IMAGES_DIR, tf)
train_floader = DataLoader(train_fds, batch_size=32, shuffle=True)
val_floader = DataLoader(val_fds, batch_size=32, shuffle=False)

img_backbone_fusion = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
fusion_model = FusionModel(img_backbone_fusion).to(device)

optimizer = torch.optim.Adam(fusion_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weight)

for epoch in range(FUSION_EPOCHS):
    fusion_model.train()
    total_loss = 0.0
    for imgs, text_emb, labels in train_floader:
        imgs, text_emb, labels = imgs.to(device), text_emb.to(device), labels.to(device)
        optimizer.zero_grad()
        out = fusion_model(imgs, text_emb)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    print(f"Epoch {epoch+1}/{FUSION_EPOCHS} - loss: {total_loss/len(train_fds):.4f}")
    torch.save(fusion_model.state_dict(), f"{DATA_DIR}/fusion_model_epoch{epoch+1}.pt")

fusion_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, text_emb, labels in val_floader:
        out = fusion_model(imgs.to(device), text_emb.to(device))
        preds = out.argmax(1).cpu().numpy()
        all_preds.extend(preds); all_labels.extend(labels.numpy())

print("=== FUSION MODEL ===")
print(classification_report(all_labels, all_preds, target_names=["compliant", "non-compliant"]))
f1_fusion = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)
print(f"F1 (non-compliant): {f1_fusion:.4f}")
save_metric("fusion_f1", f1_fusion)

torch.save(fusion_model.state_dict(), f"{DATA_DIR}/fusion_model_final.pt")
print(f"Saved -> {DATA_DIR}/fusion_model_final.pt")


## 6. Ablation summary (generated from real saved metrics, not typed in manually)

In [ ]:
m = load_metrics()
print(f"{\'Model\':<15} {\'F1 (non-compliant)\':>20}")
print("-" * 36)
print(f"{\'Text-only\':<15} {m.get(\'text_only_f1\', float(\'nan\')):>20.4f}")
print(f"{\'Image-only\':<15} {m.get(\'image_only_f1\', float(\'nan\')):>20.4f}")
print(f"{\'Fusion\':<15} {m.get(\'fusion_f1\', float(\'nan\')):>20.4f}")
